In [0]:
-- Fetch the qdp_individuals_all document data product created by 01. Quantexa Processing


-- 01. Create variables for use in this notebook
-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;



DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;




DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;



-- 2. Remove all Individual Hub and Sat records for the Tennant

-- human name prefix
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_human_name_prefix
WHERE human_name_id IN (
  SELECT hn_prefix.human_name_id
  FROM demo_banking_silver.qdp_individuals_all.sat_human_name_prefix hn_prefix
  JOIN demo_banking_silver.qdp_individuals_all.sat_human_name hn
    ON hn_prefix.human_name_id = hn.human_name_id
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual hi
    ON hn.individual_id = hi.individual_id
  WHERE hi.tennant_id = :p_tennant_id
)
;


-- human name suffix
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_human_name_suffix
WHERE human_name_id IN (
  SELECT hn_suffix.human_name_id
  FROM demo_banking_silver.qdp_individuals_all.sat_human_name_suffix hn_suffix
  JOIN demo_banking_silver.qdp_individuals_all.sat_human_name hn
    ON hn_suffix.human_name_id = hn.human_name_id
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual hi
    ON hn.individual_id = hi.individual_id
  WHERE hi.tennant_id = :p_tennant_id
);

--human names
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_human_name s
WHERE s.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;

-- individual addresses
DELETE FROM demo_banking_silver.qdp_links_all.link_individual_address_history l
WHERE l.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;

-- individual contacts
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_individual_contact s
WHERE s.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;

-- individual identifiers
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_individual_identifier s
WHERE s.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;


-- individual merged from history
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_individual_merged_from s
WHERE s.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;

-- individual communication language
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_communication_language s
WHERE s.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;

-- individual communication method
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_communication_method s
WHERE s.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;



DELETE FROM demo_banking_silver.qdp_individuals_all.sat_individual s
WHERE s.individual_id IN (
  SELECT DISTINCT individual_id
  FROM demo_banking_silver.qdp_individuals_all.hub_individual
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_silver.qdp_individuals_all.hub_individual WHERE tennant_id = :p_tennant_id;



-- 3. Create the table of individual_entity_id and associated Human Names
CREATE OR REPLACE TEMP VIEW individual_human_names AS
SELECT 
    individual_entity_id, 
    human_name.human_name.rank,
    human_name.human_name.is_trusted,
    human_name.human_name.is_preferred,
    human_name.human_name.use_code,
    try_cast(human_name.human_name.use_concept_id AS BIGINT) AS use_concept_id,
    human_name.human_name.use_raw_code,
    try_cast(human_name.human_name.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    human_name.human_name.initials,
    human_name.human_name.given1,
    human_name.human_name.given1_soundex,
    human_name.human_name.given1_raw,
    human_name.human_name.given1_raw_soundex,
    human_name.human_name.given2,
    human_name.human_name.given2_soundex,
    human_name.human_name.given2_raw,
    human_name.human_name.given2_raw_soundex,
    human_name.human_name.given3,
    human_name.human_name.given3_soundex,
    human_name.human_name.given3_raw,
    human_name.human_name.given3_raw_soundex,
    human_name.human_name.given4,
    human_name.human_name.given4_soundex,
    human_name.human_name.given4_raw,
    human_name.human_name.given4_raw_soundex,
    human_name.human_name.given5,
    human_name.human_name.given5_soundex,
    human_name.human_name.given5_raw,
    human_name.human_name.given5_raw_soundex,
    human_name.human_name.family,
    human_name.human_name.family_soundex,
    human_name.human_name.family_raw,
    human_name.human_name.family_raw_soundex,
    human_name.human_name.full_given,
    human_name.human_name.full_given_soundex,
    human_name.human_name.full_name,
    human_name.human_name.full_name_soundex,
    human_name.human_name.full_name_raw,
    human_name.human_name.full_name_raw_soundex,
    human_name.human_name.full_name_parsed,
    try_to_timestamp(human_name.human_name.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(human_name.human_name.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT ind.individual_entity_id, explode(human_names) AS human_name 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all ind
) AS exploded_names
LIMIT 1000;

SELECT * FROM individual_human_names;


-- 4. Create the table of individual_entity_id and associated Addresses
CREATE OR REPLACE TEMP VIEW individual_addresses AS
SELECT 
    individual_entity_id, 
    address.address.address_entity_id,
    address.address.rank,
    address.address.is_primary_address,
    address.address.use_code,
    try_cast(address.address.use_concept_id AS BIGINT) AS use_concept_id,
    address.address.use_raw_code,
    try_cast(address.address.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    try_to_timestamp(address.address.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(address.address.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT ind.individual_entity_id, explode(address_history) AS address 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
) AS exploded_addresses
LIMIT 1000;

SELECT * FROM individual_addresses;



-- 5. Create the table of individual_entity_id and associated Addresses
CREATE OR REPLACE TEMP VIEW exp_communication_methods AS
  SELECT ind.individual_entity_id, explode(communication_methods) AS communication_method 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
;

SELECT * FROM exp_communication_methods;


CREATE OR REPLACE TEMP VIEW individual_communication_methods AS
SELECT 
    individual_entity_id, 
    communication_method.communication_method.rank,
    communication_method.communication_method.is_primary_communication_method,
    communication_method.communication_method.value,
    communication_method.communication_method.value_display,
    communication_method.communication_method.value_raw,
    communication_method.communication_method.type_code,
    try_cast(communication_method.communication_method.type_concept_id AS BIGINT) AS type_concept_id,
    communication_method.communication_method.type_raw_code,
    try_cast(communication_method.communication_method.type_raw_concept_id AS BIGINT) AS type_raw_concept_id,
    communication_method.communication_method.use_code,
    try_cast(communication_method.communication_method.use_concept_id AS BIGINT) AS use_concept_id,
    communication_method.communication_method.use_raw_code,
    try_cast(communication_method.communication_method.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    communication_method.communication_method.telephony_country_code,
    try_cast(communication_method.communication_method.telephony_country_concept_id AS BIGINT) AS telephony_country_concept_id,
    communication_method.communication_method.telephony_country_raw_code,
    try_cast(communication_method.communication_method.telephony_country_raw_concept_id AS BIGINT) AS telephony_country_raw_concept_id,
    try_to_timestamp(communication_method.communication_method.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(communication_method.communication_method.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT ind.individual_entity_id, explode(communication_methods) AS communication_method 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
) AS exploded_communication_methods
;

SELECT * FROM individual_communication_methods;





-- 7. Create the  hub_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.hub_individual
(individual_entity_id, tennant_id, national_id, period_start, period_end)
SELECT individual_entity_id, tennant_id, national_id, period_start, period_end
FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.hub_individual;

-- 8. Create the  sat_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_individual
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
     data_source_code,
     data_source_concept_id,
     data_source_raw_code,
     data_source_raw_concept_id,
     birth_date,
     birth_datetime,
     birth_date_numeric,
     estimated_birth_date,
     estimated_birth_date_numeric,
     is_multiple_birth,
     multiple_birth_number,
     birth_place_city,
     birth_place_district,
     birth_place_country,
     birth_place_country_code,
     photo,
     is_deceased,
     deceased_datetime,
     deceased_date_numeric,
     gender_code,
     gender_concept_id,
     gender_raw_code,
     gender_raw_concept_id,
     marital_status_code,
     marital_status_concept_id,
     marital_status_raw_code,
     marital_status_raw_concept_id,
     ethnicity_code,
     ethnicity_concept_id,
     ethnicity_raw_code,
     ethnicity_raw_concept_id,
     religion_code,
     religion_concept_id,
     religion_raw_code,
     religion_raw_concept_id,
     occupation_code,
     occupation_concept_id,
     occupation_raw_code,
     occupation_raw_concept_id,
     nationality_code,
     nationality_concept_id,
     nationality_raw_code,
     nationality_raw_concept_id,
     country_of_residence_code,
     country_of_residence_concept_id,
     country_of_residence_raw_code,
     country_of_residence_raw_concept_id,
     primary_address_id,
     annual_income,
     number_of_dependants,
     graduation_date,
     is_employee,
     is_ex_employee,
     is_graduate,
     is_interpreter_required,
     is_employed,
     is_retired,
     is_student,
     is_individual_merged,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.data_source_code,
  q.data_source_concept_id,
  q.data_source_raw_code,
  q.data_source_raw_concept_id,
  q.birth_date AS birth_date,
  q.birth_datetime AS birth_datetime,
  q.birth_date_numeric AS birth_date_numeric,
  q.estimated_birth_date AS estimated_birth_date,
  q.estimated_birth_date_numeric AS estimated_birth_date_numeric,
  q.is_multiple_birth AS is_multiple_birth,
  q.multiple_birth_number AS multiple_birth_number,
  q.birth_place_city AS birth_place_city,
  q.birth_place_district AS birth_place_district,
  q.birth_place_country AS birth_place_country,
  q.birth_place_country_code AS birth_place_country_code,
  q.photo AS photo,
  q.is_deceased AS is_deceased,
  q.deceased_datetime AS deceased_datetime,
  q.deceased_date_numeric AS deceased_date_numeric,
  q.gender_code AS gender_code,
  q.gender_concept_id AS gender_concept_id,
  q.gender_raw_code AS gender_raw_code,
  q.gender_raw_concept_id AS gender_raw_concept_id,
  q.marital_status_code AS marital_status_code,
  q.marital_status_concept_id AS marital_status_concept_id,
  q.marital_status_raw_code AS marital_status_raw_code,
  q.marital_status_raw_concept_id AS marital_status_raw_concept_id,
  q.ethnicity_code AS ethnicity_code,
  q.ethnicity_concept_id AS ethnicity_concept_id,
  q.ethnicity_raw_code AS ethnicity_raw_code,
  q.ethnicity_raw_concept_id AS ethnicity_raw_concept_id,
  q.religion_code AS religion_code,
  q.religion_concept_id AS religion_concept_id,
  q.religion_raw_code AS religion_raw_code,
  q.religion_raw_concept_id AS religion_raw_concept_id,
  q.occupation_code AS occupation_code,
  q.occupation_concept_id AS occupation_concept_id,
  q.occupation_raw_code AS occupation_raw_code,
  q.occupation_raw_concept_id AS occupation_raw_concept_id,
  q.nationality_code AS nationality_code,
  q.nationality_concept_id AS nationality_concept_id,
  q.nationality_raw_code AS nationality_raw_code,
  q.nationality_raw_concept_id AS nationality_raw_concept_id,
  q.country_of_residence_code AS country_of_residence_code,
  q.country_of_residence_concept_id AS country_of_residence_concept_id,
  q.country_of_residence_raw_code AS country_of_residence_raw_code,
  q.country_of_residence_raw_concept_id AS country_of_residence_raw_concept_id,
  q.primary_address_id AS primary_address_id,
  q.annual_income AS annual_income,
  q.number_of_dependants AS number_of_dependants,
  q.graduation_date AS graduation_date,
  q.is_employee AS is_employee,
  q.is_ex_employee AS is_ex_employee,
  q.is_graduate AS is_graduate,
  q.is_interpreter_required AS is_interpreter_required,
  q.is_employed AS is_employed,
  q.is_retired AS is_retired,
  q.is_student AS is_student,
  q.is_individual_merged AS is_individual_merged,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_individual;


-- 9. Create the  sat_human_name table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
     rank,
     is_trusted,
     is_preferred,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     initials,
     given1,
     given1_soundex,
     given1_raw,
     given1_raw_soundex,
     given2,
     given2_soundex,
     given2_raw,
     given2_raw_soundex,
     given3,
     given3_soundex,
     given3_raw,
     given3_raw_soundex,
     given4,
     given4_soundex,
     given4_raw,
     given4_raw_soundex,
     given5,
     given5_soundex,
     given5_raw,
     given5_raw_soundex,
     family,
     family_soundex,
     family_raw,
     family_raw_soundex,
     full_given,
     full_given_soundex,
     full_name,
     full_name_soundex,
     full_name_raw,
     full_name_raw_soundex,
     full_name_parsed,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.rank AS rank,
  q.is_trusted AS is_trusted,
  q.is_preferred AS is_preferred,
  q.use_code,
  q.use_concept_id,
  q.use_raw_code,
  q.use_raw_concept_id,
  q.initials,
  q.given1,
  q.given1_soundex,
  q.given1_raw,
  q.given1_raw_soundex,
  q.given2,
  q.given2_soundex,
  q.given2_raw,
  q.given2_raw_soundex,
  q.given3,
  q.given3_soundex,
  q.given3_raw,
  q.given3_raw_soundex,
  q.given4,
  q.given4_soundex,
  q.given4_raw,
  q.given4_raw_soundex,
  q.given5,
  q.given5_soundex,
  q.given5_raw,
  q.given5_raw_soundex,
  q.family,
  q.family_soundex,
  q.family_raw,
  q.family_raw_soundex,
  q.full_given,
  q.full_given_soundex,
  q.full_name,
  q.full_name_soundex,
  q.full_name_raw,
  q.full_name_raw_soundex,
  q.full_name_parsed,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM individual_human_names q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name;


-- 10. Create the  sat_communication_method table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_communication_method
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
     rank,
     is_primary_communication_method,
     value,
     value_display,
     value_raw,
     type_code,
     type_concept_id,
     type_raw_code,
     type_raw_concept_id,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     telephony_country_code,
     telephony_country_concept_id,
     telephony_country_raw_code,
     telephony_country_raw_concept_id,
     period_start,
     period_end
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.rank AS rank,
  q.is_primary_communication_method AS is_primary_communication_method,
  q.value AS value,
  q.value_display AS value_display, 
  q.value_raw AS value_raw, 
  q.type_code AS type_code,
  q.type_concept_id AS type_concept_id,
  q.type_raw_code AS type_raw_code,
  q.type_raw_concept_id AS type_raw_concept_id,
  q.use_code AS use_code,
  q.use_concept_id AS use_concept_id,
  q.use_raw_code AS use_raw_code,
  q.use_raw_concept_id AS use_raw_concept_id,
  q.telephony_country_code AS telephony_country_code,
  q.telephony_country_concept_id AS telephony_country_concept_id,
  q.telephony_country_raw_code AS telephony_country_raw_code,
  q.telephony_country_raw_concept_id AS telephony_country_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM individual_communication_methods q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_communication_method;


-- 11. Create the  link_individual_address_history table records    individual_addresses
INSERT INTO demo_banking_silver.qdp_links_all.link_individual_address_history
    ( 
     individual_id, 
     individual_entity_id,
     address_id,
     address_entity_id,
     is_primary_address,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  q.individual_entity_id AS individual_entity_id,
  ha.address_id AS address_id,
  q.address_entity_id AS address_entity_id,
  q.is_primary_address AS is_primary_address,
  q.use_code AS use_code,
  q.use_concept_id AS use_concept_id,
  q.use_raw_code AS use_raw_code,
  q.use_raw_concept_id AS use_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM individual_addresses q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id
JOIN demo_banking_silver.qdp_locations_all.hub_address ha
  ON ha.address_entity_id = q.address_entity_id;

SELECT * FROM demo_banking_silver.qdp_links_all.link_individual_address_history;




